This code ingests data from bronze layer into silver and applies transformations (in this case just left join) and expectations (data quality checks).

- @pipelines.expect: monitors records, if any row doesnt match the expectation it is saved in the silver layer but it will be flagged and data quality will drop
- @pipelines.expect_or_drop: acts as a filter, if any row doesnt match the expectatin it is dropped
- @pipelines.expect_or_fail: a failsafe, raises an error if any row doesnt match the expectation. Stops the whole pipeline

In [0]:
import pyspark.sql.functions as F
from pyspark import pipelines

@pipelines.table(
    name="trezio2005_silver.silver_sales_enriched"
)
#expectations (data quality checks)
@pipelines.expect_or_drop("valid_customer", "customer_id IS NOT NULL")
@pipelines.expect("valid_loyalty_score", "loyalty_segment >= 0")
@pipelines.expect("shipment_addres_is_not_null", "ship_to_address IS NOT NULL")
def create_silver_sales():
    duplicated_colums = ['ship_to_address', 'customer_name']
    
    sales_stream = pipelines.read_stream("trezio2005_bronze.bronze_sales_orders") #streaming table with sales 
    customers_static = pipelines.read("trezio2005_bronze.bronze_customers").drop(*duplicated_colums) #normal table with customers

    return sales_stream.join(customers_static, "customer_id", "left")